# Document-Based Stores (MongoDB)

### Task 1: Create a simple MongoDB out of this relational model

This is  a toy DB about movies and actors who played roles in these movies. This DB is consisted of  

- A "Person" table who has a unique id, and a name fields.

- Another "Movie" table that has a unique id, a title, a country where it was made, and a year when it was released.

- There is (m-n) or "many-many" relationship between these two tables (i.e basically, many actors can act in many movies, and the movie include many actors)
- Therefore, we use the "Roles" table in which we can deduct which person has acted in which movie, and what role(s) they played.

<img src="figs/RDBSchema.png" alt="3" border="0">

#### Connect to the MongoDB server, and create a mongoDB with the name 'moviedb'

In [1]:
##YOUR CODE HERER
from pymongo import MongoClient
from random import randint
from pprint import pprint

import warnings
warnings.filterwarnings('ignore')

host = "mongodb://localhost:27017/", #if within docker.

# NOTE: if you are running this notebook in docker you need to 
# refer to the container name "mongodb://mongo:27017/"
# mongo = "mongo"

# we use the MongoClient to communicate with the running database instance.
myclient = MongoClient(
                    host,  
                    username='admin',
                    password='admin') #Mongo URI format
mydb = myclient["movie_db"]
# Or you can use the attribute style 
# mydb = myclient.customer_db

#### Create Person/Actor collection

In [2]:
person_coll = mydb["person"]
actor_coll = mydb["actor"]

#### Insert the data into the Person Table

In [3]:
personList = [
  { "id": 1, "name": "Charlie Sheen" },
  { "id": 2, "name": "Michael Douglas"},
  { "id": 3, "name": "Martin Sheen"},
  { "id": 4, "name": "Morgan Freeman"}
]


person_coll.insert_many(personList)


InsertManyResult([ObjectId('68e676b1013cafacf70500e8'), ObjectId('68e676b1013cafacf70500e9'), ObjectId('68e676b1013cafacf70500ea'), ObjectId('68e676b1013cafacf70500eb')], acknowledged=True)

#### Creating rest of Collections ("Movies", "Roles")

In [4]:
restcols = ["Movies","Roles"]

for col in restcols:
    mydb[col]

#### Inserting data into the movie Collection

In [5]:
movies_coll = mydb.movies

movieList = [
  { "id": 1, "title": "Wall Street", "country":"USA","year":1987},
  { "id": 2, "title": "The American President", "country":"USA","year":1995},
  { "id": 3, "title": "The Shawshank Redemption", "country":"USA","year":1994},
]

movies_coll.insert_many(movieList)

InsertManyResult([ObjectId('68e676b3013cafacf70500ec'), ObjectId('68e676b3013cafacf70500ed'), ObjectId('68e676b3013cafacf70500ee')], acknowledged=True)

#### Inserting data into the roles Collection

In [6]:
roles_col = mydb.roles

roleList = [
  { "personId": 1, "movieId": 1, "role":["Bud Fox"]},
  { "personId": 2, "movieId": 1, "role":["Carl Fox"]},
  { "personId": 3, "movieId": 1, "role":["Gordon Gekko"]},
  { "personId": 2, "movieId": 2, "role":["A.J. MacInerney"]},
  { "personId": 3, "movieId": 2, "role":["President Andrew Shepherd"]},
  { "personId": 4, "movieId": 3, "role":["Ellis Boyd 'Red' Redding"]}
]

roles_col.insert_many(roleList)

InsertManyResult([ObjectId('68e676b5013cafacf70500ef'), ObjectId('68e676b5013cafacf70500f0'), ObjectId('68e676b5013cafacf70500f1'), ObjectId('68e676b5013cafacf70500f2'), ObjectId('68e676b5013cafacf70500f3'), ObjectId('68e676b5013cafacf70500f4')], acknowledged=True)

### <font color ='green'>Just for your info</font>:

#### Another Way of Modeling this M-N model in Mongo would be using the Forien Keys 


* Movies


```[

{
	"_id": 1,
	"title":"Wall Street",
	"country":"USA",
	"year":1987,
	"persons":[1,2]
},

{
	"_id": 2,
	"title":"The American President",
	"country":"USA",
	"year":1995,
	"persons":[2]
}]
```
* Actors

```
[{
    "_id": 1,
    "name": "Charlie Sheen",
    "movies":[
    {"role": "Bud Fox", "movie_id":1}
    ]
},

{
    "_id": 2,
    "name": "Micheal Douglas",
    "movies":[
    {"role": "Gordon Geko", "movie_id":1},
    {"role": "President Andrew Shepherd", "movie_id":2}
    ]
}

] ```


#### Get all actors in your Mongo DB

In [12]:
persons = person_coll.find()
for person in persons:
    print(person)

{'_id': ObjectId('68e675c0b5184958af67b90a'), 'id': 1, 'name': 'Charlie Sheen'}
{'_id': ObjectId('68e675c0b5184958af67b90b'), 'id': 2, 'name': 'Michael Douglas'}
{'_id': ObjectId('68e675c0b5184958af67b90c'), 'id': 3, 'name': 'Martin Sheen'}
{'_id': ObjectId('68e675c0b5184958af67b90d'), 'id': 4, 'name': 'Morgan Freeman'}


#### Get actors with names start with 'C' letter

In [47]:
persons = person_coll.find({"name":{"$regex": "C"}})
for person in persons:
    print(person)

{'_id': ObjectId('68e675c0b5184958af67b90a'), 'id': 1, 'name': 'Charlie Sheen'}


#### Get all Movies sorted from recent to old! (get only the title and year fields)

In [41]:
movies_sorted = movies_coll.find().sort({"year":-1})

for movie in movies_sorted:
    print(movie)

{'_id': ObjectId('68e675e7b5184958af67b90f'), 'id': 2, 'title': 'The American President', 'country': 'USA', 'year': 1995}
{'_id': ObjectId('68e675e7b5184958af67b910'), 'id': 3, 'title': 'The Shawshank Redemption', 'country': 'USA', 'year': 1994}
{'_id': ObjectId('68e675e7b5184958af67b90e'), 'id': 1, 'title': 'Wall Street', 'country': 'USA', 'year': 1987}


#### Get all Movies released in the 90s (after year (1990) and before 2000) ordered from old to recent.

In [45]:
movies_started_in_90 = movies_coll.find({"$and": [ {"year": {"$gte": 1990} },
                                                  {"year": {"$lte": 2000}}
                            ]}).sort({"year":1})

for movie in movies_started_in_90:
    print(movie)

{'_id': ObjectId('68e675e7b5184958af67b910'), 'id': 3, 'title': 'The Shawshank Redemption', 'country': 'USA', 'year': 1994}
{'_id': ObjectId('68e675e7b5184958af67b90f'), 'id': 2, 'title': 'The American President', 'country': 'USA', 'year': 1995}


#### Get Movies and Actors from your "movies" DB
* Hint : use the <code>'$lookup'</code> operator.
* The Result should be something like the following:
<code>
Charlie Sheen : Wall Street  
Michael Douglas : Wall Street  
Martin Sheen : Wall Street  
Michael Douglas : The American President  
Martin Sheen : The American President  
Morgan Freeman : The Shawshank Redemption  
</code>

In [61]:
actors_and_movie = roles_col.aggregate([
  {
    "$lookup": {
      "from": "person",
      "localField": "personId",
      "foreignField": "id",
      "as": "person"
    }
  },
  { "$unwind": "$person" },
  {
    "$lookup": {
      "from": "movies",
      "localField": "movieId",
      "foreignField": "id",
      "as": "movie"
    }
  },
  { "$unwind": "$movie" },
  {
    "$project": {
      "_id": 0,
      "actor": "$person.name",
      "title": "$movie.title"
    }
  }
])

for ac in actors_and_movie:
    print(ac['actor'], " : ", ac['title'])

Charlie Sheen  :  Wall Street
Michael Douglas  :  Wall Street
Martin Sheen  :  Wall Street
Michael Douglas  :  The American President
Martin Sheen  :  The American President
Morgan Freeman  :  The Shawshank Redemption


#### For each Actor, get count of "Movies" he acted in.

In [67]:
actors_and_movie = roles_col.aggregate([
  {
    "$lookup": {
      "from": "person",
      "localField": "personId",
      "foreignField": "id",
      "as": "person"
    }
  },
  { "$unwind": "$person" },
  {  "$group": {"_id":"$person.name",
                  "myCount": { '$sum': 1 } }
    },
  {
    "$project": {
      "_id": 0,
      "actor": "$_id",
      "numberOfMovies": "$myCount"
    }
  }
])

for ac in actors_and_movie:
    print(ac['actor'], " : ", ac['numberOfMovies'])


Morgan Freeman  :  1
Michael Douglas  :  2
Martin Sheen  :  2
Charlie Sheen  :  1


#### In your DB, list the movies that every Actor played

In [ ]:
actors_and_movie = roles_col.aggregate([
  {
    "$lookup": {
      "from": "person",
      "localField": "personId",
      "foreignField": "id",
      "as": "person"
    }
  },
  { "$unwind": "$person" },
  {
    "$lookup": {
      "from": "movies",
      "localField": "movieId",
      "foreignField": "id",
      "as": "movie"
    }
  },
  { "$unwind": "$movie" },
  {  "$group": {"_id":"$person.name",
                  "movies": { "$addToSet": "$movie.title" } }
  },
  {
    "$project": {
      "_id": 0,
      "actor": "$_id",
      "movies": 1
    }
  }
])

for ac in actors_and_movie:
    print(ac['actor'], " : ", ac['movies'])

Michael Douglas  :  ['Wall Street', 'The American President']
Martin Sheen  :  ['The American President', 'Wall Street']
Charlie Sheen  :  ['Wall Street']
Morgan Freeman  :  ['The Shawshank Redemption']


#### Get the Persons/Actors who acted in "Wall Street" movie
- Hint use `$lookup` , `$match` operators in the aggregation piepeline

In [ ]:
###YOUR CODE HERE

#### Get the Movies in which "Micheal Douglas" has played a role in

In [ ]:
###YOUR CODE HERE

#### Get count of "Movies" in your DB

In [ ]:
###YOUR CODE HERE

#### update the year of the 'Wall Street' movie was released in to be 2000(which is not true BTW :)
- Show that movie before and After updating it

In [ ]:
###YOUR CODE HERE

####  Delete all the persons with names start with 'M' letter.

In [ ]:
###YOUR CODE HERE

### Task 2: Extend your Mongo-"MovieDB" 

Imagine now that we are going to extend our DB with new movies, actors, even with new directors.

- We add <b>**"The matrix"**</b> movie which was released in <b> USA, (1999)</b>, and has a new property/field "Tagline" <b>("Welcome to the Real World")</b>.
 
- We will also add 4 new actors (Person):
    - **"Keanu Reeves"** who was born in (1964). <font color='green'>Note:</font> "born" property is also new.
    - **"Carrie-Anne Moss"** who was born in (1967).
    - **"Laurence Fishburne"** who was born in (1960).
    - **"Hugo Weaving"** who was born in (1960).
    
- Moreover, we add 2 directors (Person) :
    - **"Lilly Wachowski"**, born in (1967)
    - **"Lana Wachowski"**, born in(1965)
- For these directors specify one more label/field as ("Director"). (You can add this while inserting the director documents)
    
 
- We will also create a new <b>collection "Directed" </b> that is directed from the later 2 directors to "the Matrix" movie.

#### Add the Movie "The Matrix" with the provided data to the Movies collection

In [ ]:
###YOUR CODE HERE

#### Insert the new 4 actors to the person collection

In [ ]:
#Notice, How is easy to add a new feild compared to the RDB
newActorList = [
  { "id": 5, "name": "Keanu Reeves", "born":1964 },
  { "id": 6, "name": "Carrie-Anne Moss", "born":1967},
  { "id": 7, "name": "Laurence Fishburne", "born":1960},
  { "id": 8, "name": "Hugo Weaving", "born":1960}
]

###YOUR CODE HERE

#### Insert the new 2 directors to the person collection

In [ ]:
###YOUR CODE HERE

#### Create the "Directed" collection, and insert the data into it 

In [ ]:
###YOUR CODE HERE

#### Get only the directors from the person collection (i.e. persons marked with the label "Director")

In [ ]:
###YOUR CODE HERE

#### Perform a query that get persons (names, and born year) who Directed "The Matrix" movie.

In [ ]:
###YOUR CODE HERE

 ## How long did it take you to solve the homework?
 
Please answer as precisely as you can. It does not affect your points or grade in any way. It is okey, if it took 0.5 hours or 24 hours. The collected information will be used to improve future homeworks.

<font color="red"><b>Answer:</b></font>

**<center> <font color='red'>THANK YOU FOR YOUR EFFORT!</font></center>**